# Direct Lyapunov Safety-Gate RL Study (Cold Start)

This notebook matches the pretrained safety-gate study, but each method starts from a fresh untrained TD3 agent.

In [ ]:
import os
from datetime import datetime
from pprint import pprint

import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

import torch

In [ ]:
from TD3Agent.agent import TD3Agent
from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from Simulation.mpc import compute_observer_gain
from Simulation.run_rl_lyapunov import run_rl_train
from Simulation.system_functions import PolymerCSTR
from Lyapunov.direct_lyapunov_mpc import design_direct_lyapunov_mpc_solver
from Lyapunov.safety_debug import (
    build_safety_filter_run_bundle,
    make_safety_filter_comparison_record,
    save_safety_filter_comparison_artifacts,
    save_safety_filter_debug_artifacts,
)
from utils.direct_lyapunov_study import (
    DIRECT_DISTURBANCE_N_TESTS,
    DIRECT_DISTURBANCE_SEED,
    DIRECT_DISTURBANCE_SETPOINT_LEN,
    DIRECT_TWO_SETPOINT_Y_PHYS,
    direct_disturbance_test_cycle,
    direct_four_method_case_specs,
)
from utils.scaling_helpers import apply_min_max
from utils.td3_helpers import load_and_prepare_system_data

predict_h = 9
cont_h = 3
rho_lyap = 0.98
lyap_eps = 1e-9
lyap_tol = 1e-10
slack_penalty = 1e6
terminal_cost_scale = 1.0
objective_steady_input_cost = False
objective_terminal_cost = False
plant_mode = "disturb"
disturbance_after_step = False

Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

steady_states = {"ss_inputs": system_steady_state_inputs.copy()}
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)
steady_states["y_ss"] = cstr_ss.y_ss.copy()

u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
setpoint_y_phys = DIRECT_TWO_SETPOINT_Y_PHYS.copy()

n_tests = DIRECT_DISTURBANCE_N_TESTS
set_points_len = DIRECT_DISTURBANCE_SETPOINT_LEN
TEST_CYCLE = direct_disturbance_test_cycle(n_tests)
warm_start = 0
WARMUP_EPISODES = 10
BC_TEACHER_EPISODES = 20
time_in_sub_episodes = setpoint_y_phys.shape[0] * set_points_len
ACTOR_FREEZE = 0 * set_points_len
warm_start_plot = np.array(
    [
        WARMUP_EPISODES * time_in_sub_episodes,
        (WARMUP_EPISODES + BC_TEACHER_EPISODES) * time_in_sub_episodes,
    ],
    dtype=int,
)
training_phase_config = {
    "episode_unit": "cycle",
    "warmup_buffer_only_episodes": WARMUP_EPISODES,
    "behavior_clone_teacher_episodes": BC_TEACHER_EPISODES,
    "exploration_std_start": 0.02,
    "exploration_std_end": 0.0,
    "exploration_decay_scope": "entire_run",
    "bc_teacher_policy": "direct_lyapunov_mpc",
    "warmup_behavior_source": "direct_lyapunov_mpc",
}

study_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(os.path.join(os.getcwd(), "Data", "debug_exports"), exist_ok=True)

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y_phys,
    u_min=u_min,
    u_max=u_max,
    system_dict_path=os.path.join("Data", "system_dict"),
    augmentation_style="rawlings",
    augmentation_mode="output_disturbance",
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(setpoint_y_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"],
    data_min[inputs_number:],
    data_max[inputs_number:],
)

poles = np.array([
    0.44619852,
    0.33547649,
    0.36380595,
    0.70467118,
    0.3562966,
    0.42900673,
    0.4228262,
    0.96916776,
    0.91230187,
])
L = compute_observer_gain(A_aug, C_aug, poles)

set_points_number = int(C_aug.shape[0])
STATE_DIM = int(A_aug.shape[0]) + set_points_number + inputs_number
ACTION_DIM = int(B_aug.shape[1])
ACTOR_LAYER_SIZES = [512, 512, 512, 512, 512]
CRITIC_LAYER_SIZES = [512, 512, 512, 512, 512]
BUFFER_CAPACITY = 40000
ACTOR_LR = 5e-5
CRITIC_LR = 5e-4
SMOOTHING_STD = 0.005
NOISE_CLIP = 0.01
GAMMA = 0.995
TAU = 0.005
MAX_ACTION = 1
POLICY_DELAY = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256
STD_START = 0.02
STD_END = 0.0
STD_DECAY_RATE = 0.99992
STD_DECAY_MODE = "exp"

Qy_diag = np.array([5.0, 1.0])
Su_diag = np.array([1.0, 1.0])
Rdu_diag = np.array([1.0, 1.0])
reward_config, reward_fn = make_reward_fn_mpc_quadratic(Q_diag=Qy_diag, R_diag=Rdu_diag)

u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:inputs_number], data_max[:inputs_number])
b_min = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
b_max = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])
b1 = (b_min[0] - u_ss[0], b_max[0] - u_ss[0])
b2 = (b_min[1] - u_ss[1], b_max[1] - u_ss[1])
bnds = (b1, b2) * cont_h
IC_opt_template = np.zeros(inputs_number * cont_h)

u_min_scaled = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
u_max_scaled = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])
u_dev_min = u_min_scaled - u_ss
u_dev_max = u_max_scaled - u_ss

MPC_obj = design_direct_lyapunov_mpc_solver(
    A_aug=A_aug,
    B_aug=B_aug,
    C_aug=C_aug,
    Qy_diag=Qy_diag,
    NP=predict_h,
    NC=cont_h,
    Su_diag=Su_diag,
    u_min=u_dev_min,
    u_max=u_dev_max,
    Rdu_diag=Rdu_diag,
    terminal_set_on=True,
    terminal_alpha_scale=1.0,
    terminal_cost_scale=terminal_cost_scale,
    objective_steady_input_cost=objective_steady_input_cost,
    objective_terminal_cost=objective_terminal_cost,
)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92

case_specs = direct_four_method_case_specs()


def make_td3_agent():
    return TD3Agent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=ACTOR_LAYER_SIZES,
        critic_hidden=CRITIC_LAYER_SIZES,
        gamma=GAMMA,
        actor_lr=ACTOR_LR,
        critic_lr=CRITIC_LR,
        batch_size=BATCH_SIZE,
        policy_delay=POLICY_DELAY,
        target_policy_smoothing_noise_std=SMOOTHING_STD,
        noise_clip=NOISE_CLIP,
        max_action=MAX_ACTION,
        tau=TAU,
        std_start=STD_START,
        std_end=STD_END,
        std_decay_rate=STD_DECAY_RATE,
        std_decay_mode=STD_DECAY_MODE,
        buffer_size=BUFFER_CAPACITY,
        device=DEVICE,
        actor_freeze=ACTOR_FREEZE,
    )

In [ ]:
study_name = "rl_direct_safety_gate_four_method_two_setpoint_disturb_cold_start"
study_root = os.path.join(os.getcwd(), "Data", "debug_exports", study_name, study_timestamp)
os.makedirs(study_root, exist_ok=True)


def run_case(case_spec):
    case_name = case_spec["case_name"]
    case_target_config = dict(case_spec.get("target_config", {}))
    case_agent = make_td3_agent()
    cstr_case = PolymerCSTR(
        system_params,
        system_design_params,
        system_steady_state_inputs,
        delta_t,
        deviation_form=False,
    )

    case_config = {
        "study_name": study_name,
        "case_name": case_name,
        "projection_backend": "direct_accept_or_fallback",
        "target_mode": case_spec["target_mode"],
        "target_config": case_target_config,
        "rho_lyap": rho_lyap,
        "lyap_eps": lyap_eps,
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "disturbance_after_step": disturbance_after_step,
        "training_phase_config": dict(training_phase_config),
        "initial_agent_path": None,
    }

    results_case = run_rl_train(
        system=cstr_case,
        y_sp_scenario=y_sp_scenario,
        n_tests=n_tests,
        set_points_len=set_points_len,
        steady_states=steady_states,
        min_max_dict=min_max_dict,
        agent=case_agent,
        MPC_obj=MPC_obj,
        L=L,
        data_min=data_min,
        data_max=data_max,
        warm_start=warm_start,
        test_cycle=TEST_CYCLE,
        nominal_qi=nominal_qi,
        nominal_qs=nominal_qs,
        nominal_ha=nominal_hA,
        qi_change=qi_change,
        qs_change=qs_change,
        ha_change=ha_change,
        reward_fn=reward_fn,
        mode=plant_mode,
        rho_lyap=rho_lyap,
        lyap_eps=lyap_eps,
        lyap_tol=lyap_tol,
        seed=DIRECT_DISTURBANCE_SEED,
        use_lyap=True,
        IC_opt=IC_opt_template.copy(),
        bnds=bnds,
        cons=(),
        reuse_mpc_solution_as_ic=False,
        reset_system_on_entry=True,
        projection_backend="direct_accept_or_fallback",
        first_step_contraction_on=True,
        direct_target_mode=case_spec["target_mode"],
        direct_target_config=case_target_config,
        direct_tracking_use_target_output=False,
        disturbance_after_step=disturbance_after_step,
        training_phase_config=training_phase_config,
    )

    bundle_case = build_safety_filter_run_bundle(
        source=case_name,
        results=results_case,
        steady_states=steady_states,
        config=case_config,
        min_max_dict=min_max_dict,
        data_min=data_min,
        data_max=data_max,
        extra={
            "delta_t": delta_t,
            "warm_start_plot": warm_start_plot,
            "start_plot_idx": 10,
            "agent_path": None,
            "reward_config": reward_config,
            "actor_losses": case_agent.actor_losses,
            "critic_losses": case_agent.critic_losses,
        },
    )
    debug_dir_case = save_safety_filter_debug_artifacts(
        bundle=bundle_case,
        directory=study_root,
        prefix_name=case_name,
        save_plots=True,
        save_paper_plots=True,
        save_rl_summary_plots=True,
    )
    trained_agent_path = case_agent.save(debug_dir_case, prefix="trained_agent", include_optim=False)
    record_case = make_safety_filter_comparison_record(case_name, bundle_case, debug_dir_case)
    record_case["trained_agent_path"] = trained_agent_path
    print(f"Completed {case_name}")
    pprint(record_case)
    return bundle_case, debug_dir_case, record_case


bundles_by_case = {}
debug_dirs_by_case = {}
comparison_records = []
for case_spec in case_specs:
    bundle_case, debug_dir_case, record_case = run_case(case_spec)
    case_name = case_spec["case_name"]
    bundles_by_case[case_name] = bundle_case
    debug_dirs_by_case[case_name] = debug_dir_case
    comparison_records.append(record_case)

comparison_artifacts = save_safety_filter_comparison_artifacts(
    comparison_records,
    bundles_by_case,
    study_root,
    save_plots=True,
)
print("Saved cold-start RL direct-gate study:")
pprint(comparison_artifacts)
comparison_df = pd.DataFrame(comparison_records) if pd is not None else comparison_records
comparison_df